# Ray checkpointing example

In this notebook, we will go through setting up and running a Ray Train checkpointing demo on Red Hat OpenShift AI, with the following steps:
 * Defining a RayCluster configuration
 * Setting up AWS credentials for S3 bucket to store and retrieve checkpointing data
 * Starting a Train Job with checkpointing enabled
 * Confirming that the checkpoitning data for the Train Job can be persisted and Train Job can be restored from previously saved checkpoint

## Import required libraries

In [ ]:
from codeflare_sdk import Cluster, ClusterConfiguration, RayJob, set_api_client
from kube_authkit import AuthConfig, get_k8s_client
import time
import urllib3

## Authenticate to your OpenShift cluster

In [ ]:
# Authenticate to your Kubernetes/OpenShift cluster using kube-authkit

# Option 1: Auto-detect credentials (kubeconfig or in-cluster service account)
# NOTE: In RHOAI Workbenches the workbench service account may not have Ray RBAC
# permissions. Use Option 2 (token) unless your admin has granted SA permissions
# (see RHOAIENG-46748). Auto-detect works if you have a local kubeconfig.
# auth_config = AuthConfig(method="auto")

# Option 2 (Recommended for RHOAI Workbenches): Token-based authentication
# Get your token with: oc whoami -t  (or from the OpenShift console → Copy login command)
auth_config = AuthConfig(
    method="openshift",
    k8s_api_host="https://api.example.com:6443",
    token="sha256~XXXXX",  # oc whoami -t
)

# Option 3: OIDC authentication (for BYOIDC-enabled clusters)
# auth_config = AuthConfig(
#     method="oidc",
#     k8s_api_host="https://api.example.com:6443",
#     oidc_issuer="https://your-oidc-provider.com",
#     client_id="your-client-id",
#     use_device_flow=True,  # Interactive device flow for notebook environments
# )

api_client = get_k8s_client(config=auth_config)
set_api_client(api_client)

## Define and start your RayCluster

In [ ]:
NAMESPACE = "your-namespace"  # Your Data Science Project namespace
CLUSTER_NAME = "checkpoint-demo-cluster" # Your RayCluster name
cluster = Cluster(ClusterConfiguration(
    name=CLUSTER_NAME,
    namespace=NAMESPACE,
    num_workers=2,
    head_cpu_requests=2,
    head_cpu_limits=4,
    head_memory_requests=4,
    head_memory_limits=8,
    worker_cpu_requests=2,
    worker_cpu_limits=4,
    worker_memory_requests=4,
    worker_memory_limits=8,
))

cluster.apply()
cluster.wait_ready()
print(f"Cluster ready: {cluster.cluster_uri()}")

## Set your AWS credentials

In [ ]:
# Set your AWS credentials
# WARNING: Do not commit credentials to version control. For production,
# use RHOAI Data Connections or OpenShift Secrets instead.
AWS_CREDENTIALS = {
    "AWS_ACCESS_KEY_ID": "your-access-key",
    "AWS_SECRET_ACCESS_KEY": "your-secret-key",
    "AWS_DEFAULT_REGION": "your-region" # e.g. "us-east-1"
    "AWS_S3_BUCKET": "your-bucket-name",
    # "AWS_SESSION_TOKEN": "your-session-token", # optional - if using temporary AWS credentials
}

# If using temporary credentials (SSO/federated), add the session token:
# AWS_CREDENTIALS["AWS_SESSION_TOKEN"] = "your-session-token"

print(f"Using bucket: {AWS_CREDENTIALS['AWS_S3_BUCKET']}")

## Submit initial RayJob

In [ ]:
job = RayJob(
    job_name="checkpointing-job",
    cluster_name=CLUSTER_NAME,
    namespace=NAMESPACE,
    entrypoint="python train_with_checkpoints.py",
    runtime_env={
        "working_dir": "./",
        "pip": ["torch", "torchvision", "s3fs", "pyarrow"],
        "env_vars": {
            **AWS_CREDENTIALS,
            "RAY_TRAIN_WORKER_GROUP_START_TIMEOUT_S": "120", # Allow time for worker scheduling
        },
    },
)

job.submit()
print("Job submitted - watch for 'NO CHECKPOINT FOUND - Starting fresh' in logs")

## Monitor Job Progress

In [ ]:
# Check job status
"""
Check job status and access the Ray dashboard for detailed logs.

Open the Ray dashboard URL in your browser and navigate to the **Jobs** tab. Watch for:
- `NO CHECKPOINT FOUND - Starting fresh training from epoch 0` (first run)
- `Checkpoint saved to S3` messages after each epoch

Wait until you see at least one checkpoint saved before proceeding to the next cell.
"""
job.status()

# Get the Ray dashboard URL to view logs
print(f"Ray Dashboard: {cluster.cluster_dashboard_uri()}")
print("Open the dashboard and go to the Jobs tab to view logs")

## Delete the Job
Delete the existing job, once a checkpoint was successfully saved to s3 bucket

In [ ]:
print("=" * 60)
print("DELETING JOB - Simulating stop/preemption")
print("Checkpoints are persisted in S3!")
print("=" * 60)

try:
    job.delete()
    print("Job deleted")
except:
    print("Job already deleted or not found")

time.sleep(5)
print("Ready to resubmit and demonstrate checkpoint resume")

## Re-submit RayJob

In [ ]:
print("=" * 60)
print("RESUBMITTING JOB")
print("Watch for 'RESUMING FROM CHECKPOINT' in logs!")
print("=" * 60)

job = RayJob(
    job_name="checkpointing-job",
    cluster_name=CLUSTER_NAME,
    namespace=NAMESPACE,
    entrypoint="python train_with_checkpoints.py",
    runtime_env={
        "working_dir": "./",
        "pip": ["torch", "torchvision", "s3fs", "pyarrow"],
        "env_vars": {
            **AWS_CREDENTIALS,
            "RAY_TRAIN_WORKER_GROUP_START_TIMEOUT_S": "120",
        },
    },
)

job.submit()
print("Job resubmitted - waiting for logs...")
time.sleep(45)

## Verify Resume from Checkpoint
In the Ray dashboard Jobs tab, look for the success message:
```
RESUMING FROM CHECKPOINT - Starting at epoch N
Previous loss: X.XXXX
```

This confirms the model state, optimizer state, and training progress were preserved across the job restart.

In [ ]:
job.status()

# View logs in Ray dashboard
print(f"Ray Dashboard: {cluster.cluster_dashboard_uri()}")
print("Check the Jobs tab for: 'RESUMING FROM CHECKPOINT - Starting at epoch N'")

## Cleanup

In [ ]:
print("Cleaning up...")
try:
    job.delete()
except:
    pass

try:
    cluster.down()
except:
    pass

print("Cluster and job cleaned up")

In [ ]:
# No explicit logout needed - authentication is managed automatically by kube-authkit